# CardioTrust: selective cardiovascular-label prediction

Byte2Beat research prototype. This notebook reproduces the saved experiment and evidence. It is not a diagnostic tool.

## Publication gate

The primary dataset's upstream Kaggle Data Card reports its license as `Unknown`. Keep the notebook private and do not redistribute attached row-level data until the owner or host provides a valid license.

In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import Image, Markdown, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA = ROOT / 'data' / 'raw' / 'cardio_base.csv'
ARTIFACTS = ROOT / 'artifacts'
assert DATA.exists(), 'Place the host-provided cardio_base.csv in data/raw/'

## Data audit

The identifier is excluded from modeling. Every record belonging to a duplicated predictor profile is removed before splitting. Plausibility rules are fixed without using the target, while imputation is learned inside training folds.

In [ ]:
raw = pd.read_csv(DATA, sep=';')
profile_columns = [column for column in raw.columns if column not in ('id', 'cardio')]
duplicated_profile = raw.duplicated(profile_columns, keep=False)
duplicated_groups = raw.loc[duplicated_profile].groupby(profile_columns, dropna=False)
audit = pd.Series({
    'rows': len(raw),
    'columns': raw.shape[1],
    'positive_prevalence': raw['cardio'].mean(),
    'missing_values': raw.isna().sum().sum(),
    'duplicate_feature_target_rows_after_first': raw.drop(columns='id').duplicated().sum(),
    'duplicated_profile_rows_excluded': int(duplicated_profile.sum()),
    'duplicated_profile_groups': duplicated_groups.ngroups,
    'conflicting_label_groups': int((duplicated_groups['cardio'].nunique() > 1).sum()),
})
display(audit.to_frame('value'))

## Leakage-safe experiment

The protocol and code hashes were frozen before the final holdout was observed. The script uses an 80/20 stratified development/locked-holdout split. Logistic regression and histogram gradient boosting are compared with 3 repeats of 5-fold out-of-fold predictions; each training fit contains three-fold sigmoid calibration. Model selection and the abstention threshold use development data only.

In [ ]:
# Run training in an isolated process so notebook kernels do not retain workers.
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'src.run_experiment'], cwd=ROOT, check=True)

In [ ]:
metrics = json.loads((ARTIFACTS / 'metrics.json').read_text())
comparison = pd.DataFrame(metrics['development']).T[
    ['roc_auc', 'average_precision', 'balanced_accuracy', 'brier', 'ece_10']
]
display(Markdown(f"**Selected model:** `{metrics['selected_model']}`"))
display(comparison.style.format('{:.4f}'))
display(pd.Series(metrics['holdout_at_0_5'], name='holdout').to_frame())
display(pd.DataFrame(metrics['holdout_intervals_95']).T)

## Selective prediction and subgroup evidence

Coverage thresholds are learned from development out-of-fold confidence and applied unchanged to the holdout. Subgroup results are descriptive and are not evidence of fairness or clinical equivalence.

In [ ]:
selective = pd.read_csv(ARTIFACTS / 'selective_prediction.csv')
subgroups = pd.read_csv(ARTIFACTS / 'subgroup_audit.csv')
display(selective)
display(subgroups)

In [ ]:
for figure in ('roc_pr.png', 'calibration.png', 'coverage_risk.png', 'holdout_intervals.png', 'subgroup_performance.png', 'permutation_importance.png'):
    display(Image(filename=str(ARTIFACTS / 'figures' / figure), width=760))

## Limitations

- The dataset license, collection sites, dates, consent basis, and label construction are unresolved.
- There is no external, temporal, geographic, hospital, or prospective validation.
- Source age-60-plus performance is materially weaker than the aggregate result.
- Confidence is model confidence, not patient-specific medical certainty.
- Do not use CardioTrust for diagnosis, treatment, triage, or emergency decisions.